In [2]:
# Imports
from typing import Literal, List, Union, Optional, Annotated
import time

# Dependencies
import requests
import pandas as pd
import mwparserfromhell
from pydantic import BaseModel, Field

# Definition
JSONType = dict[str]

In [3]:
# Classes
class	MediaWikiAPI:
	"""
	Class that contains the url for API request. It exist to make HTTP requests only.
	"""
	API_URL = "https://cardfight.fandom.com/api.php"

	def	__init__(self):
		self.session = requests.Session()

	def	get(self,
			params: dict[str, Union[str, List[str]]],
			headers: dict[str, str]) -> JSONType:
		"""
		Function to obtain information from the MediaWikiAPI. In order to use it, you
		must define ther correct HTTP parameter. The returned data will have a json structure.

		Parameters:
			params: necesary parameters to make a request to the API. Please consult https://www.mediawiki.org/wiki/API:Action_API.
			headers: HTTP headers (such as User-Agent).

		Returns:
			JSONType: If the request was succesful, you will have a json file with the desired information.
		"""
		return (
			self.session.get(	
			self.API_URL,
			params=params,
			headers=headers
			).json()
		)
	
class	VanguardScrapper:
	def	__init__(self, api: MediaWikiAPI):
		self.api = api
	
	def	obtain_main_sets(self, data: JSONType):
		sets = []
		for link in data["parse"]["links"]:
			if (link.get("ns") == 0):
				sets.append(link["*"])
		return (sets)
	
	def	separate_boosters(self, data: list):
		no_main_sets = []
		for set in range(len(data) - 1, -1, -1):
			value = data[set]
			if (("Booster" not in value) or ("Cardfight!!" in value) or ("Unique" in value)):
				no_main_sets.append(data.pop(set))
		return (no_main_sets)

	def remove_from_list(self, sets: list, to_delete: list):
		for i in to_delete:
			if (i in sets):
				sets.remove(i)
	
	def	inner_url_sets(self, sets: list):
		inner_categories = {
			"limit_break":	[],
			"stride":		[],
			"v":			[],
			"d":			[],
			"dz":			[]
		}
		url = {
			"limit_break":	[],
			"stride":		[],
			"v":			[],
			"d":			[],
			"dz":			[]
		}
		

class	ScrapCard(BaseModel):
	card_nro:		str
	name:			str
	grade:			Optional[int] = None
	nation:			str
	card_type:		str | None = None
	rarity:			str
	release:		str

class	Card(BaseModel):
	name:			str
	kana:			Optional[str] = None
	phonetic:		Optional[str] = None
	thai:			Optional[str] = None
	card_type:		str
	grade:			Optional[int] = None
	skill:			Optional[str] = None
	power:			Optional[int] = None
	shield:			Optional[int] = None
	critical:		Optional[int] = None
	nation:			Optional[str] = None
	race:			Optional[str] = None
	valid_format:	Optional[str] = None
	card_set:		list[str] = Field(default_factory=list)
	card_flavor:	Optional[str] = None
	card_effect:	str
	tournament:		Optional[str] = None

class	Trigger(Card):
	card_type:		str = "trigger_unit"
	boost:			Optional[int] = None
	trigger_effect:	str

class	Order(Card):
	card_type:		str = "order"
	order_type:		str

class	NormalOrder(Order):
	order_type:		Literal["normal"]

class	BlitzOrder(Order):
	order_type:		Literal["blitz"]

CardUnion = Annotated[
	Union[Trigger, NormalOrder, BlitzOrder],
	Field(discriminator="card_type")
]

In [ ]:
# Params
titles = []
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
params = {
	"action": "query",
	"format": "json",
	"prop": "revisions",
	"title": titles,
	"rvprop": "content",
	"rvslots": "main"
}
params_for_urls = {
	"action": "parse",
	"page": "List_of_Cardfight!!_Vanguard_Booster_Sets",
	"format": "json"
}

In [ ]:
# Url Parser
web = MediaWikiAPI()
scrap = VanguardScrapper(web)
urls = scrap.api.get(params=params_for_urls, headers=header)
sets = scrap.obtain_main_sets(urls)
no_main_sets = scrap.separate_boosters(sets)
scrap.remove_from_list(no_main_sets, ["Lyrical Monasterio", "The Mask Collection"])
scrap.remove_from_list(sets, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
])

In [ ]:
sets

In [ ]:
sets_cpy = sets.copy()

In [ ]:
for i in sets_cpy:
	i.replace()

In [ ]:
no_main_sets

In [ ]:
inner_categories = {
	"limit_break":	{},
	"stride":		{},
	"v":			{},
	"d":			{},
	"dz":			{}
}

def	obtain_url(func):
	def wrapper(s: str):
		result = func(s)
		return (result.replace(" ", "_"))
	return (wrapper)

@obtain_url
def	obtain_limit_break(s: str):
	s_set = "LB " + s
	inner_categories["limit_break"]['set'] = s_set
	inner_categories["limit_break"]['url'] = s
	return (s)

@obtain_url
def	obtain_legion(s: str):
	pass

@obtain_url
def	obtain_stride(s: str):
	pass

@obtain_url
def	obtain_v(s: str):
	pass

@obtain_url
def	obtain_d(s: str):
	pass

@obtain_url
def	obtain_dz(s: str):
	pass

functions = {
	obtain_limit_break,
	obtain_legion,
	obtain_stride,
	obtain_v,
	obtain_d,
	obtain_dz
}


In [ ]:
x = sets[0]

In [ ]:
x

In [ ]:
s = obtain_limit_break(x)

In [ ]:
s